# XoFTR instead of LoFTR in the fine-matching step (both with MINIMA weights)

The fine stage takes each of the 25 coarse candidates, matches the north-up canvas against the map window of that candidate, fits one transform with RANSAC and reports **inliers** (how sure) and a **shift** (how exact). Today the matcher is LoFTR (`minima_loftr.ckpt`, coarse threshold 0.1, confidence cut 0.1 after `LoFTR_More_Matches`). Here XoFTR (`minima_xoftr.ckpt`) takes its place.

The swap is one block: `src/fine_matching/xoftr_matcher.py` (`load_xoftr_matcher`) returns the same `LoftrMatcher` object around an XoFTR model, so RANSAC, the shift rule and `run_fine_stage` are the pipeline's own and unchanged. The model code is `third_party/XoFTR` (inference-only copy).

**What is compared** (same photos, same coarse candidates, same RANSAC 3 px, nothing else changed): LoFTR as it was, LoFTR as chosen now, and XoFTR at its MINIMA defaults and two looser settings. For every photo we take the candidate with most inliers, answer if it reaches the bar, and score it with the **real scoring rule** applied to the error *after the shift* (`header.scoring.points_for_error`: +10 within 1 sigma, +6, +3, 0, -20). So the table measures both the decision evidence (inliers) and the exactness of the shift.

Candidates per photo: up to 3 right places (coarse peak within 4 sigma of the truth) and the 24 best-ranked wrong peaks (the look-alikes that hurt). Photos with no right place stay in the count: answering them costs -20. Photos are drawn with a seed so other seeds and the NUDT set can be run by changing the settings cell (or the env vars `XL_DATA`, `XL_SEED`).

In [ ]:
import os, sys, math, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2, torch
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

# ------------------------------------------------------------------ settings
DATA = os.environ.get("XL_DATA", "official")      # "official" (d1 photos on the new map) or "nudt" (Changsha test photos)
SEED = int(os.environ.get("XL_SEED", "45"))        # picks which photos are checked
MAP = "new"                 # base map of the official photos
MAP_MARGIN_M = 2500
N_PHOTOS = 500              # photos loaded
N_CHECK = int(os.environ.get("XL_N", "200"))      # photos checked
K_PEAKS = 25
N_RIGHT = 3                 # right candidates kept per photo
N_WRONG = 24                # best-ranked wrong candidates per photo
FOUND_SIGMA = 4.0
RANSAC_PIXELS = 3.0
BARS = [8, 12]              # answer bars (pipeline: 12)
# the variants: name, family, coarse threshold, fine threshold (XoFTR only), confidence cut used when fitting
VARIANTS = [
    dict(name="LoFTR old",            family="loftr", coarse=0.2, fine=None, cut=0.3),
    dict(name="LoFTR chosen",         family="loftr", coarse=0.1, fine=None, cut=0.1),
    dict(name="XoFTR default",        family="xoftr", coarse=0.3, fine=0.1,  cut=0.1),
    dict(name="XoFTR coarse 0.2",     family="xoftr", coarse=0.2, fine=0.1,  cut=0.1),
    dict(name="XoFTR loose 0.1/0.05", family="xoftr", coarse=0.1, fine=0.05, cut=0.05),
]
plt.rcParams["figure.dpi"] = 90
print("data", DATA, "| seed", SEED, "| photos checked", N_CHECK)

## 1 · Set up the pipeline (coarse search + map) and pick the candidates

In [ ]:
from single_photo_pipeline import SinglePhotoPipeline
from header import settings
from header.scoring import points_for_error
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, map_piece
from train.coarse_methods.inputs import make_inputs
from semantic_rerank.dino_rerank import coarse_top
from fine_matching.loftr_matcher import loftr_matches, fit_matches, prepare_loftr_canvas, reliable_mask
from header.settings import CTX

pipeline = SinglePhotoPipeline(DATA, "fused", semantic_layer=False, k_peaks=K_PEAKS, m_fine=K_PEAKS, matcher="loftr_minima",
                               satellite=MAP if DATA == "official" else None, map_margin_m=MAP_MARGIN_M if DATA == "official" else None,
                               ransac_pixels=RANSAC_PIXELS)
start = time.time()
pipeline.setup(N_PHOTOS, 7 if DATA == "nudt" else SEED)
print("setup %.0f s" % (time.time() - start))
test_set, camera, level_images = pipeline.test_set, pipeline.camera, pipeline.level_images

order = np.sort(np.random.default_rng(SEED).choice(len(test_set.items), N_CHECK, replace=False))
cases = []
for index in order:
    item = test_set.items[index]
    truth, sigma = test_set.truths[index], test_set.sigmas[index]
    level = level_for_height(item["rel_height"], camera)
    canvas, valid = render_query(item, camera, level)
    cv2.setRNGSeed(settings.SEED)
    inputs = make_inputs(canvas, valid, level, test_set.priors[index], pipeline.families, pipeline.names)
    peaks = coarse_top(pipeline.coarse_method(inputs), inputs["geometry"], K_PEAKS)
    del inputs
    errors = np.array([math.hypot(p[1] - truth[0], p[2] - truth[1]) / sigma if p[1] < 1e8 else 1e9 for p in peaks])
    right = [k for k in range(len(peaks)) if errors[k] <= FOUND_SIGMA][:N_RIGHT]
    wrong = [k for k in range(len(peaks)) if errors[k] > FOUND_SIGMA and peaks[k, 1] < 1e8][:N_WRONG]
    picture, inside = prepare_loftr_canvas(canvas, valid)
    cases.append(dict(has_right=len(right) > 0, photo=int(index), height=item["rel_height"], level=level, sigma=sigma, truth=truth, picture=picture,
                      reliable=reliable_mask(inside, 0),
                      candidates=[dict(x=peaks[k, 1], y=peaks[k, 2], role="right") for k in right] + [dict(x=peaks[k, 1], y=peaks[k, 2], role="wrong") for k in wrong]))
    torch.cuda.empty_cache()
print("%d photos: %d have a right place among the %d coarse peaks, %d have none (kept: every answer there is wrong); %d candidates" % (
    len(cases), sum(c["has_right"] for c in cases), K_PEAKS, sum(not c["has_right"] for c in cases), sum(len(c["candidates"]) for c in cases)))

## 2 · Load the matchers and run every variant on every candidate

Raw matches (positions and confidence) are stored, so the confidence cut and RANSAC can be changed afterwards without running the network again. Time per pair is the network only (synchronised).

In [ ]:
from fine_matching.loftr_matcher import load_loftr_matcher
from fine_matching.xoftr_matcher import load_xoftr_matcher
checkpoints = ROOT / "dataset" / "NUDT" / "checkpoint" / "MINIMA"
if not checkpoints.exists():
    from train import config
    checkpoints = config.find_nudt_dir() / "checkpoint" / "MINIMA"

loftr = load_loftr_matcher(checkpoints / "minima_loftr.ckpt")
xoftr = load_xoftr_matcher(checkpoints / "minima_xoftr.ckpt")
print("parameters: LoFTR %.1f M, XoFTR %.1f M" % (sum(p.numel() for p in loftr.loftr_model.parameters()) / 1e6, sum(p.numel() for p in xoftr.loftr_model.parameters()) / 1e6))

def set_thresholds(variant):
    if variant["family"] == "loftr":
        loftr.loftr_model.coarse_matching.thr = variant["coarse"]
        return loftr
    xoftr.loftr_model.xoftr.coarse_matching.thr = variant["coarse"]
    xoftr.loftr_model.xoftr.fine_matching.fine_thr = variant["fine"]
    return xoftr

raw, seconds = {}, {}
for variant in VARIANTS:
    matcher = set_thresholds(variant)
    stored, pairs, start = {}, 0, time.time()
    for case in cases:
        for position, cand in enumerate(case["candidates"]):
            tile = map_piece(level_images, case["level"], cand["x"], cand["y"], CTX)
            stored[(case["photo"], position)] = loftr_matches(matcher, case["picture"], tile)
            pairs += 1
    if settings.DEVICE == "cuda":
        torch.cuda.synchronize()
    raw[variant["name"]] = stored
    seconds[variant["name"]] = (time.time() - start) / pairs
    print("%-22s %d pairs, %.3f s/pair, %.0f matches/pair" % (variant["name"], pairs, seconds[variant["name"]], np.mean([len(v[2]) for v in stored.values()])))

## 3 · Read every variant with the pipeline's own rule

Per variant: fit RANSAC on the stored matches, take per photo the candidate with the most inliers, answer when it reaches the bar, score the corrected position with the real points rule.

* **right: median inliers** (best right candidate) and **wrong: 95th pct** (best wrong candidate): how well the inlier count separates right from wrong;
* **answered right / WRONG**: share of all checked photos (WRONG = answered but more than 4 sigma off, mostly -20);
* **median error after shift (sigma)** of the chosen answers that are right: the exactness of the refinement;
* **points per photo**: mean of the real points (-20 .. +10) over all checked photos, skips count 0.

In [ ]:
def fit_all(name, cut, px=RANSAC_PIXELS):
    out = {}
    for case in cases:
        for position, cand in enumerate(case["candidates"]):
            pa, pb, conf = raw[name][(case["photo"], position)]
            cv2.setRNGSeed(settings.SEED)
            result = fit_matches(pa, pb, conf, case["reliable"], case["level"], cut, px)
            error_sigma = math.hypot(cand["x"] + result["shift"][0] - case["truth"][0], cand["y"] + result["shift"][1] - case["truth"][1]) / case["sigma"]
            out[(case["photo"], position)] = (result["n_inliers"], error_sigma)
    return out


def photo_outcomes(fits, bar):
    """One entry per photo: (answered, points, error in sigma of the answer, best right inliers, best wrong inliers)."""
    rows = []
    for case in cases:
        values = [fits[(case["photo"], k)] for k in range(len(case["candidates"]))]
        roles = [c["role"] for c in case["candidates"]]
        best = max(range(len(values)), key=lambda k: values[k][0])             # most inliers wins (first one on a tie)
        right_inliers = [values[k][0] for k in range(len(values)) if roles[k] == "right"]
        wrong_inliers = [values[k][0] for k in range(len(values)) if roles[k] == "wrong"]
        answered = values[best][0] >= bar
        points = 0
        if answered:
            points = points_for_error(values[best][1] * case["sigma"], case["height"], camera.fx)
        rows.append((answered, points, values[best][1], max(right_inliers) if right_inliers else -1, max(wrong_inliers) if wrong_inliers else 0))
    return rows


def read_variant(name, cut, px=RANSAC_PIXELS):
    fits = fit_all(name, cut, px)
    row = {"variant": name, "cut": cut, "ransac px": px, "s/pair": seconds[name]}
    first = photo_outcomes(fits, BARS[0])
    best_right = np.array([r[3] for r in first]); best_wrong = np.array([r[4] for r in first]); has_right = best_right >= 0
    row["right: median inliers"] = np.median(best_right[has_right])
    row["wrong: 95th pct"] = np.percentile(best_wrong, 95)
    per_photo = {}
    for bar in BARS:
        outcomes = photo_outcomes(fits, bar)
        answered = np.array([o[0] for o in outcomes]); points = np.array([o[1] for o in outcomes], dtype=float); error = np.array([o[2] for o in outcomes])
        right_answer = answered & (error <= FOUND_SIGMA)
        row["answered right @%d" % bar] = right_answer.mean()
        row["answered WRONG @%d" % bar] = (answered & ~right_answer).mean()
        row["median error @%d (sigma)" % bar] = np.median(error[right_answer]) if right_answer.any() else np.nan
        row["within 1 sigma @%d" % bar] = (answered & (error <= 1.0)).mean()
        row["points @%d" % bar] = points.mean()
        per_photo[bar] = points
    return row, per_photo


rows, points_by_variant = [], {}
for variant in VARIANTS:
    row, per_photo = read_variant(variant["name"], variant["cut"])
    rows.append(row); points_by_variant[variant["name"]] = per_photo
table = pd.DataFrame(rows)
print("%d photos (%d with a right place), %d candidates | RANSAC %.0f px | pipeline bar = 12" % (len(cases), sum(c["has_right"] for c in cases), sum(len(c["candidates"]) for c in cases), RANSAC_PIXELS))
display(table.round(3))

## 4 · Inliers of right and wrong candidates (the separation the answer bar relies on)

In [ ]:
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(4.2 * len(VARIANTS), 3.4), sharey=True)
for axis, variant in zip(axes, VARIANTS):
    fits = fit_all(variant["name"], variant["cut"])
    right_values = [fits[(c["photo"], k)][0] for c in cases for k, cand in enumerate(c["candidates"]) if cand["role"] == "right"]
    wrong_values = [fits[(c["photo"], k)][0] for c in cases for k, cand in enumerate(c["candidates"]) if cand["role"] == "wrong"]
    bins = np.linspace(0, max(right_values + wrong_values + [10]), 40)
    axis.hist(wrong_values, bins=bins, alpha=0.6, color="tab:gray", label="wrong", density=True)
    axis.hist(right_values, bins=bins, alpha=0.6, color="tab:green", label="right", density=True)
    for bar in BARS:
        axis.axvline(bar, color="red", ls="--", lw=0.8)
    axis.set_yscale("log"); axis.set_title(variant["name"], fontsize=9); axis.set_xlabel("RANSAC inliers")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5 · Is the difference real? Paired bootstrap over photos (points per photo, XoFTR minus LoFTR chosen)

The photos are the same for both matchers, so the difference is taken per photo and resampled. An interval that excludes 0 means the gap is not luck of the photo draw. (Other seeds and the NUDT set are separate runs of this notebook: see the result files.)

In [ ]:
def bootstrap_diff(a, b, n=4000, seed=0):
    rng = np.random.default_rng(seed)
    diff = a - b
    means = np.array([diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n)])
    return diff.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

reference = "LoFTR chosen"
rows = []
for variant in VARIANTS:
    if variant["name"] == reference:
        continue
    for bar in BARS:
        mean, low, high = bootstrap_diff(points_by_variant[variant["name"]][bar], points_by_variant[reference][bar])
        rows.append({"variant": variant["name"], "bar": bar, "points diff vs %s" % reference: mean, "95% low": low, "95% high": high, "excludes 0": (low > 0) or (high < 0)})
bootstrap_table = pd.DataFrame(rows)
display(bootstrap_table.round(3))

## 6 · Confidence cut and RANSAC distance on the stored matches (no network run)

In [ ]:
names = ["LoFTR chosen", "XoFTR default"]
rows = []
for name in names:
    for cut in (0.05, 0.1, 0.2, 0.3, 0.5):
        for px in (1.5, 3.0, 5.0):
            if name.startswith("LoFTR") and cut < 0.1:
                continue                                            # LoFTR itself drops matches below its coarse threshold
            row, _ = read_variant(name, cut, px)
            rows.append(row)
sweep = pd.DataFrame(rows)
for bar in BARS:
    print("points per photo @%d (rows: confidence cut, columns: RANSAC px)" % bar)
    display(sweep.pivot_table(index=["variant", "cut"], columns="ransac px", values="points @%d" % bar).round(2))
out_dir = ROOT / "tests" / "Test_Iteration_4" / "results" / "xoftr_vs_loftr"
out_dir.mkdir(parents=True, exist_ok=True)
tag = "%s_seed%d" % (DATA, SEED)
table.to_csv(out_dir / ("variants_%s.csv" % tag), index=False)
bootstrap_table.to_csv(out_dir / ("bootstrap_%s.csv" % tag), index=False)
sweep.to_csv(out_dir / ("cut_ransac_sweep_%s.csv" % tag), index=False)
print("saved in", out_dir)